# vigilex – Notebook 03
## Feature Engineering & Modellierung

**Voraussetzung:** Notebook 02 wurde ausgeführt → `data/processed/maude_labeled.parquet` existiert.

**Ziel dieses Notebooks:**
1. Sinnvolle Features aus den MAUDE-Rohdaten bauen
2. Baseline-Modell (Logistic Regression) trainieren und evaluieren
3. LightGBM-Modell trainieren (das finale Modell für die API)
4. Modell als `.pkl` speichern für den FastAPI-Endpoint

---

### Warum diese Reihenfolge?

Feature Engineering vor Modellierung ist nicht nur Konvention – es ist inhaltlich notwendig:
Die Rohdaten (MAUDE-Records) repräsentieren einzelne Meldungen.
Ein Recall betrifft aber immer ein **Gerät** oder einen **Hersteller** über eine **Zeitspanne**.
Wir müssen also von Meldungs-Ebene auf Geräte/Zeitfenster-Ebene aggregieren.
Das ist der Kern des Feature Engineerings hier.

---
## 📖 How to Read This Notebook (Beginner Guide)

This is **Notebook 03** — the feature engineering and machine learning modelling step.
It was part of the original plan (before the architecture shifted to the PostgreSQL-native
MedDRA coding pipeline). It shows how you would build a recall prediction model
using classical ML on the MAUDE data.

### Key concepts explained here

**Feature Engineering**
: Transforming raw data into informative inputs for a machine learning model.
  Raw data (dates, free text, product codes) cannot be fed directly into most ML algorithms —
  you need to create numerical representations. Examples here:
  - Severity score from event type categories
  - Rolling window complaint counts (how many reports in the last 30/90 days?)
  - Text features from narrative keywords

**Rolling Window** (time series)
: A sliding time window that computes statistics over a recent period.
  Example: "How many adverse event reports were filed for this device in the last 90 days?"
  This captures trend information that a single report cannot.
  Window sizes used: 30, 90, 180 days.

**LightGBM** (Gradient Boosting)
: A fast, high-performance machine learning algorithm for tabular data.
  It builds an ensemble of decision trees iteratively, where each new tree
  corrects the errors of the previous ones ("gradient boosting").
  Chosen over Random Forest because: faster, handles imbalanced data better,
  built-in support for categorical features.

**Class Imbalance**
: In the recall dataset, only ~5% of devices get recalled. If we trained naively,
  the model would just predict "not recalled" for everything and be 95% accurate —
  but completely useless. `scale_pos_weight` in LightGBM corrects for this by
  giving more weight to the rare positive class.

**Train/Test Split** (time-based)
: We split data by date rather than randomly. Records before 2023 = training set;
  records from 2023 onwards = test set. This simulates real deployment:
  we train on historical data and predict on future data.
  A random split would leak future information into training (data leakage).

**SHAP Values** (model explainability)
: SHapley Additive exPlanations — a method from game theory that computes
  how much each feature contributed to each individual prediction.
  Important for regulatory contexts: we need to be able to explain WHY the model
  flagged a particular device, not just THAT it did.

**ROC-AUC** vs **PR-AUC**:
- ROC-AUC: how well does the model separate positives from negatives overall?
- PR-AUC: how well does it find the rare positives (recalls) without too many false alarms?
  PR-AUC is more informative for imbalanced datasets (where positives are rare).

### Note: Architecture context
This notebook represents the ORIGINAL plan (classical ML recall prediction).
The CURRENT architecture uses a different approach: MedDRA coding (Module 2) +
PRR/ROR signal detection (Module 3 — not yet built). Notebook 02 (recall label join)
will eventually validate whether Module 3 signals could have predicted actual recalls.
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..') if Path('..').joinpath('data').exists() else Path('.')

df = pd.read_parquet(ROOT / 'data/processed/maude_labeled.parquet')
df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')
df['date_of_event'] = pd.to_datetime(df['date_of_event'], errors='coerce')

print(f'Records: {len(df):,}')
print(f'Spalten: {df.columns.tolist()}')
print(f'\nLabel-Verteilung:')
print(df[['recalled_ever', 'recalled_within_12m']].apply(lambda c: c.value_counts()).T)

## 1. Feature Engineering

### 1a. Severity Score

**Warum?** `event_type` und `patient_outcome` sind kategorisch und ungleich gewichtet.
Ein Death ist regulatorisch anders zu bewerten als eine Malfunction ohne Patientenschaden.
Wir bauen einen ordinalen Score, der das klinische Gewicht eines Events quantifiziert.
Dieser Score ist domänenexpertenbasiert – keine ML-Magie, sondern medizinisches Urteil.

Das ist wichtig für die spätere Erklärbarkeit: Ein Auditor versteht 'Severity Score 4 (Death)'
besser als einen abstrakten Feature-Importance-Wert.

In [ ]:
# Severity-Mapping: klinisch begründet, nicht willkürlich
# Quelle: FDA MDR 21 CFR 803 Klassifikation + EU MDR Anhang XIV
EVENT_TYPE_SCORE = {
    'Death':       5,
    'Injury':      3,
    'Malfunction': 1,
    'Other':       1,
    '':            1,
}

OUTCOME_SCORE = {
    'Death':                   5,
    'Life Threatening':        4,
    'Hospitalization':         3,
    'Required Intervention':   2,
    'Disability':              2,
    'Congenital Anomaly':      2,
    'Other':                   1,
    '':                        0,
}

df['event_score']   = df['event_type'].map(EVENT_TYPE_SCORE).fillna(1)
df['outcome_score'] = df['patient_outcome'].map(OUTCOME_SCORE).fillna(0)

# Kombinierter Severity Score: max(), nicht sum(), weil wir den schlimmsten Fall wollen
df['severity_score'] = df[['event_score', 'outcome_score']].max(axis=1)

print('Severity Score Verteilung:')
print(df['severity_score'].value_counts().sort_index())

# Plausibilitätscheck: Deaths müssen Score 5 haben
assert df.loc[df['event_type'] == 'Death', 'severity_score'].min() == 5, 'Death-Score falsch!'
print('✓ Plausibilitätsprüfung OK')

### 1b. Rolling Window Features

**Warum?** Ein einzelner MAUDE-Report sagt wenig. Aber 50 Reports für dasselbe Gerät
innerhalb von 3 Monaten ist ein starkes Warnsignal – genau das, was Recall-Entscheidungen auslöst.

Rolling Windows aggregieren die Meldungshistorie pro Gerät über Zeitfenster.
Das ist das Herzstück des Frühwarnsystems.

**Technische Umsetzung:** `product_code` als Geräte-Proxy (einfacher als brand_name,
weil Schreibweise oft inkonsistent). Wir verwenden pandas `.rolling()` mit
`on='date_received'` für zeitbasierte Fenster.

In [ ]:
# Sortieren für Rolling-Berechnung (zwingend erforderlich)
df = df.sort_values(['product_code', 'date_received']).reset_index(drop=True)

def add_rolling_features(df, group_col, date_col, windows_days):
    """
    Rolling-Window Features pro Gruppe.
    
    Berechnet für jede Zeitfenster-Größe:
    - Anzahl Reports (Complaint Volume)
    - Mittlerer Severity Score
    - Anzahl schwerer Events (Score >= 3)
    """
    df = df.copy()
    
    for window in windows_days:
        w_str = f'{window}d'
        
        # Rolling pro Gruppe
        rolled = (
            df.groupby(group_col, group_keys=False)
            .apply(lambda g: g.set_index(date_col)['severity_score']
                   .rolling(w_str, closed='left')  # closed='left': aktueller Report zählt nicht mit
                   .agg(['count', 'mean'])
                   .rename(columns={'count': f'reports_{w_str}', 'mean': f'severity_mean_{w_str}'})
                   .reset_index())
        )
        
        df[f'reports_{w_str}']       = rolled[f'reports_{w_str}'].values
        df[f'severity_mean_{w_str}'] = rolled[f'severity_mean_{w_str}'].values
        
        # Anteil schwerer Events im Fenster
        severe = (
            df.groupby(group_col, group_keys=False)
            .apply(lambda g: g.set_index(date_col)
                   .assign(is_severe=lambda x: (x['severity_score'] >= 3).astype(int))['is_severe']
                   .rolling(w_str, closed='left').mean()
                   .reset_index())
        )
        df[f'severe_rate_{w_str}'] = severe['is_severe'].values
        
    return df

# 3 Zeitfenster: 90 Tage (kurzfristig), 180 Tage (mittelfristig), 365 Tage (langfristig)
print('Berechne Rolling-Window Features (dauert ~1-2 Minuten)...')
df = add_rolling_features(df, 'product_code', 'date_received', [90, 180, 365])

rolling_cols = [c for c in df.columns if 'reports_' in c or 'severity_mean_' in c or 'severe_rate_' in c]
print(f'\n{len(rolling_cols)} Rolling Features erstellt:')
print(rolling_cols)
print(f'\nBeispiel (erste 5 Rows):')
print(df[['product_code', 'date_received'] + rolling_cols].head())

### 1c. Zeitliche Features

**Warum?** Recall-Risiken sind nicht gleichmäßig über die Zeit verteilt.
Neue Geräte auf dem Markt haben eine andere Risikodynamik als etablierte Produkte.
Saisonale Effekte (z.B. mehr Reports im Winter bei Infusionspumpen in Kliniken)
können schwaches Signal liefern.

Außerdem: Der EU AI Act verlangt Transparenz über zeitliche Einschränkungen des Modells.
Wenn wir 'Jahr' als Feature verwenden, müssen wir das im Modell-Steckbrief dokumentieren.

In [ ]:
df['year']  = df['date_received'].dt.year
df['month'] = df['date_received'].dt.month
df['quarter'] = df['date_received'].dt.quarter

# Tage seit erstem Report für dieses Gerät (Proxy für 'Gerät ist schon lange auf dem Markt')
first_report = df.groupby('product_code')['date_received'].transform('min')
df['days_since_first_report'] = (df['date_received'] - first_report).dt.days

print('Zeitliche Features:')
print(df[['year', 'month', 'quarter', 'days_since_first_report']].describe())

### 1d. Text-Feature aus Narrative

**Warum?** Die Narrative-Texte enthalten klinische Details, die strukturierte Felder nicht erfassen.
Vollständiges NLP (Embeddings, BERT) ist für das Basismodell überdimensioniert –
aber ein paar einfache Text-Features geben dem Modell ein Signal aus dem Freitext.

Für Phase II (RAG) werden die Narratives direkt als Kontext genutzt.
Hier reichen einfache Heuristiken.

In [ ]:
# Keyword-Sets: regulatorisch relevante Begriffe
HIGH_RISK_KEYWORDS = [
    'death', 'died', 'fatal', 'fatality',
    'overdose', 'occlusion', 'no delivery', 'no insulin',
    'software', 'firmware', 'update', 'defect',
    'recall', 'corrective', 'field safety',
]

def text_features(text):
    if not isinstance(text, str) or len(text) < 5:
        return 0, 0, 0
    text_lower = text.lower()
    n_words    = len(text.split())
    n_keywords = sum(1 for kw in HIGH_RISK_KEYWORDS if kw in text_lower)
    # Normalisiert: Keywords pro 100 Wörter
    keyword_density = n_keywords / max(n_words, 1) * 100
    return n_words, n_keywords, keyword_density

text_feat = df['narrative_text'].apply(text_features)
df['narrative_length']          = text_feat.apply(lambda x: x[0])
df['narrative_keyword_count']   = text_feat.apply(lambda x: x[1])
df['narrative_keyword_density'] = text_feat.apply(lambda x: x[2])

print('Text-Feature Statistiken:')
print(df[['narrative_length', 'narrative_keyword_count', 'narrative_keyword_density']].describe().round(2))

# Beispiel: Reports mit hoher Keyword-Dichte
print('\nBeispiel: Report mit hoher Keyword-Dichte:')
high_kw = df.nlargest(1, 'narrative_keyword_count')
print(high_kw['narrative_text'].values[0][:300])

### 1e. Kategorische Features encodieren

**Warum LabelEncoder statt OneHot?** LightGBM kann mit kategorischen Codes
direkt arbeiten (native categorical support) – kein Memory-Overhead durch
One-Hot-Spalten. Für Logistic Regression würde OneHot besser passen,
aber dort nutzen wir nur numerische Features.

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['product_code', 'event_type', 'report_source', 'reporter_country']

le_map = {}
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].fillna('UNKNOWN').astype(str))
    le_map[col] = le
    print(f'{col}: {le.classes_[:5]}... ({len(le.classes_)} Klassen)')

print('\n✓ Encoding fertig')

## 2. Feature-Set zusammenstellen

Wir arbeiten mit zwei Targets:
- `recalled_ever` → einfacher, für Baseline
- `recalled_within_12m` → realistischer, für das finale Modell

**Wichtiger Hinweis zu Data Leakage:**
Die Rolling-Window Features sind mit `closed='left'` berechnet –
der aktuelle Report zählt nicht mit in sein eigenes Fenster.
Damit vermeiden wir, dass das Modell die Antwort bereits in den Features hat.
Das ist bei Zeitreihendaten ein häufiger und gefährlicher Fehler.

In [ ]:
FEATURE_COLS = [
    # Severity
    'severity_score', 'event_score', 'outcome_score',
    # Rolling Windows
    'reports_90d', 'reports_180d', 'reports_365d',
    'severity_mean_90d', 'severity_mean_180d', 'severity_mean_365d',
    'severe_rate_90d', 'severe_rate_180d', 'severe_rate_365d',
    # Zeitlich
    'year', 'month', 'quarter', 'days_since_first_report',
    # Text
    'narrative_length', 'narrative_keyword_count', 'narrative_keyword_density',
    # Kategorisch (encoded)
    'product_code_enc', 'event_type_enc', 'report_source_enc', 'reporter_country_enc',
]

TARGET_SIMPLE  = 'recalled_ever'
TARGET_FINAL   = 'recalled_within_12m'

# Fehlende Werte durch 0 ersetzen (Rolling-Features haben NaN am Anfang)
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

X = df[FEATURE_COLS]
y_simple = df[TARGET_SIMPLE]
y_final  = df[TARGET_FINAL]

print(f'Feature Matrix: {X.shape}')
print(f'\nTarget recalled_ever:       {y_simple.sum():,} positiv ({y_simple.mean():.1%})')
print(f'Target recalled_within_12m: {y_final.sum():,} positiv ({y_final.mean():.1%})')

## 3. Train/Test Split

**Warum zeitbasierter Split, kein zufälliger?**

Bei Zeitreihendaten ist ein zufälliger Train/Test-Split falsch –
er würde dem Modell erlauben, zukünftige Informationen zu sehen.
In der Praxis würde das zu massiv überschätzter Performance führen.

Wir splitten nach Datum: Alles bis Ende 2022 = Training, ab 2023 = Test.
Das simuliert, wie das Modell in der Realität eingesetzt würde:
auf historischen Daten trainiert, auf neuen Daten evaluiert.

In [ ]:
SPLIT_DATE = '2023-01-01'

train_mask = df['date_received'] < SPLIT_DATE
test_mask  = df['date_received'] >= SPLIT_DATE

X_train, X_test = X[train_mask], X[test_mask]
y_train_s, y_test_s = y_simple[train_mask], y_simple[test_mask]
y_train_f, y_test_f = y_final[train_mask],  y_final[test_mask]

print(f'Training:  {len(X_train):,} Records (bis {SPLIT_DATE})')
print(f'Test:      {len(X_test):,} Records (ab {SPLIT_DATE})')
print(f'\nTrain Recall-Rate (ever):     {y_train_s.mean():.1%}')
print(f'Test  Recall-Rate (ever):     {y_test_s.mean():.1%}')
print(f'Train Recall-Rate (12m):      {y_train_f.mean():.1%}')
print(f'Test  Recall-Rate (12m):      {y_test_f.mean():.1%}')

## 4. Baseline: Logistic Regression auf `recalled_ever`

**Warum Baseline?**
Bevor wir ein komplexes Modell bauen, brauchen wir einen Vergleichswert.
Wenn LightGBM nicht deutlich besser als Logistic Regression ist,
ist das entweder ein Zeichen für gutes Feature Engineering
oder schlechte Datenqualität – beides ist wichtig zu wissen.

**Class Imbalance:**
Bei stark unausgewogenen Klassen (viele 0, wenige 1) ist Accuracy irreführend –
ein Modell, das immer 0 vorhersagt, hat hohe Accuracy aber keinen Nutzen.
Wir schauen deshalb auf **AUC-ROC** und **Average Precision (PR-AUC)**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# Pipeline: Skalierung + Modell (Logistic Regression braucht standardisierte Features)
baseline_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',  # Kompensiert Class Imbalance
        max_iter=500,
        random_state=42
    ))
])

baseline_pipe.fit(X_train, y_train_s)
y_pred_baseline    = baseline_pipe.predict(X_test)
y_proba_baseline   = baseline_pipe.predict_proba(X_test)[:, 1]

auc_baseline = roc_auc_score(y_test_s, y_proba_baseline)
pr_baseline  = average_precision_score(y_test_s, y_proba_baseline)

print(f'=== Baseline: Logistic Regression (recalled_ever) ===')
print(f'AUC-ROC:          {auc_baseline:.4f}')
print(f'PR-AUC:           {pr_baseline:.4f}')
print(f'\n{classification_report(y_test_s, y_pred_baseline)}')

## 5. Hauptmodell: LightGBM auf `recalled_within_12m`

**Warum LightGBM?**
- Sehr effizient auf tabellarischen Daten (schneller als XGBoost)
- Native kategorische Feature-Unterstützung
- `scale_pos_weight` für Class Imbalance
- Feature Importance out-of-the-box (wichtig für Erklärbarkeit)
- Gut deploybar als `.pkl`

**Target:** `recalled_within_12m` ist das realistischere Label:
"Hat dieses Gerät innerhalb von 12 Monaten nach diesem Report einen Recall gehabt?"
Das ist das eigentliche Frühwarnsystem-Signal.

In [ ]:
import lightgbm as lgb

# scale_pos_weight: Verhältnis Negativ zu Positiv (kompensiert Imbalance)
pos = y_train_f.sum()
neg = len(y_train_f) - pos
scale_pw = neg / pos if pos > 0 else 1.0
print(f'scale_pos_weight: {scale_pw:.1f} ({neg:,} negativ / {pos:,} positiv)')

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=50,      # Overfitting-Schutz
    scale_pos_weight=scale_pw,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Early Stopping: stoppt wenn Validation-AUC 20 Runden lang nicht besser wird
# Verhindert Overfitting ohne Grid Search
callbacks = [lgb.early_stopping(stopping_rounds=20, verbose=False),
             lgb.log_evaluation(period=50)]

lgb_model.fit(
    X_train, y_train_f,
    eval_set=[(X_test, y_test_f)],
    eval_metric='auc',
    callbacks=callbacks
)

y_pred_lgb   = lgb_model.predict(X_test)
y_proba_lgb  = lgb_model.predict_proba(X_test)[:, 1]

auc_lgb = roc_auc_score(y_test_f, y_proba_lgb)
pr_lgb  = average_precision_score(y_test_f, y_proba_lgb)

print(f'\n=== LightGBM (recalled_within_12m) ===')
print(f'AUC-ROC:          {auc_lgb:.4f}  (Baseline: {auc_baseline:.4f})')
print(f'PR-AUC:           {pr_lgb:.4f}')
print(f'\n{classification_report(y_test_f, y_pred_lgb)}')

## 6. Modell-Evaluation

### Warum ROC + PR Kurve zusammen?

**ROC-Kurve** zeigt die Trade-off zwischen True Positive Rate und False Positive Rate.
AUC-ROC = 1.0 ist perfekt, 0.5 = zufällig.

**Precision-Recall-Kurve** ist bei starker Class Imbalance informativer:
Sie zeigt, wie viele der vorhergesagten Recalls wirklich Recalls sind (Precision)
vs. wie viele echte Recalls wir finden (Recall).
Ein Frühwarnsystem will hohen Recall (keine echten Recalls verpassen),
auch wenn dabei mehr False Positives entstehen.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Kurve
fpr, tpr, _ = roc_curve(y_test_f, y_proba_lgb)
axes[0].plot(fpr, tpr, lw=2, label=f'LightGBM (AUC={auc_lgb:.3f})')
axes[0].plot([0,1],[0,1], 'k--', label='Zufall')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Kurve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall Kurve
prec, rec, _ = precision_recall_curve(y_test_f, y_proba_lgb)
axes[1].plot(rec, prec, lw=2, color='orange', label=f'LightGBM (PR-AUC={pr_lgb:.3f})')
baseline_pr = y_test_f.mean()
axes[1].axhline(baseline_pr, ls='--', color='grey', label=f'Zufall ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Kurve')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Feature Importance
importance = pd.Series(
    lgb_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True).tail(15)

importance.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 15 Feature Importance (LightGBM)')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig(ROOT / 'models/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ models/evaluation_plots.png gespeichert')

## 7. Modell erklären: SHAP Values

**Warum SHAP?** Feature Importance sagt welche Features wichtig sind – aber nicht warum.
SHAP (SHapley Additive exPlanations) zeigt für jeden einzelnen Report:
"Dieser Report hat einen hohen Risk Score, weil reports_365d=47 (+0.23 SHAP)
und severity_score=5 (+0.18 SHAP)." 

Das ist regulatorisch entscheidend: Der EU AI Act (Art. 13) verlangt Transparenz
und Erklärbarkeit für Hochrisiko-KI-Systeme. SHAP ist die Standardmethode dafür.

In [ ]:
try:
    import shap
    
    # SHAP TreeExplainer ist für Baummodelle optimiert (sehr schnell)
    explainer = shap.TreeExplainer(lgb_model)
    
    # Sample: 500 Test-Records für SHAP (sonst zu langsam)
    X_sample = X_test.sample(min(500, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(X_sample)
    
    # Bei Binär-Klassifikation: shap_values[1] = Klasse 1 (Recall)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    plt.sca(axes[0])
    shap.summary_plot(sv, X_sample, feature_names=FEATURE_COLS, show=False, max_display=12)
    axes[0].set_title('SHAP Summary Plot')
    
    plt.sca(axes[1])
    shap.summary_plot(sv, X_sample, feature_names=FEATURE_COLS, 
                      plot_type='bar', show=False, max_display=12)
    axes[1].set_title('SHAP Feature Importance (mean |SHAP|)')
    
    plt.tight_layout()
    plt.savefig(ROOT / 'models/shap_plots.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('→ models/shap_plots.png gespeichert')
    
except ImportError:
    print('shap nicht installiert. pip install shap')
    print('SHAP wird für die Streamlit-Demo benötigt – bitte installieren.')

## 8. Modell speichern

Wir speichern:
1. Das LightGBM-Modell als `.pkl` (für FastAPI)
2. Die Feature-Liste (damit der API-Endpoint weiß welche Features er erwartet)
3. Den LabelEncoder-Map (für kategorische Features in neuen Daten)
4. Modell-Metadaten als JSON (für Transparenz / EU AI Act Dokumentation)

In [ ]:
import joblib
import json
from datetime import datetime

model_dir = ROOT / 'models'
model_dir.mkdir(exist_ok=True)

# 1. Modell
joblib.dump(lgb_model, model_dir / 'lgb_recall_risk.pkl')
print('→ models/lgb_recall_risk.pkl gespeichert')

# 2. Feature-Liste
joblib.dump(FEATURE_COLS, model_dir / 'feature_cols.pkl')
print('→ models/feature_cols.pkl gespeichert')

# 3. LabelEncoder-Map
joblib.dump(le_map, model_dir / 'label_encoders.pkl')
print('→ models/label_encoders.pkl gespeichert')

# 4. Modell-Metadaten (EU AI Act Art. 13: Transparenz)
metadata = {
    'model_name':       'vigilex-lgb-v1',
    'model_type':       'LightGBMClassifier',
    'target':           TARGET_FINAL,
    'training_cutoff':  SPLIT_DATE,
    'trained_on':       datetime.now().isoformat(),
    'n_training_records': int(len(X_train)),
    'n_test_records':   int(len(X_test)),
    'auc_roc':          round(auc_lgb, 4),
    'pr_auc':           round(pr_lgb, 4),
    'positive_rate_train': round(float(y_train_f.mean()), 4),
    'positive_rate_test':  round(float(y_test_f.mean()), 4),
    'device_categories':   df['product_code'].unique().tolist(),
    'features':         FEATURE_COLS,
    'risk_classification': 'HIGH_RISK_AI_SYSTEM',  # EU AI Act Annex III
    'intended_use': 'Recall risk prediction for drug delivery medical devices (FDA MAUDE data)',
    'limitations': [
        'Trained only on insulin pump and related device data',
        f'Training data until {SPLIT_DATE}',
        'Recall labels from FDA Recall DB only (US market)',
        'Not validated on EU EUDAMED data'
    ]
}

with open(model_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('→ models/model_metadata.json gespeichert')
print(f'\n=== Finale Modell-Performance ===')
print(f'AUC-ROC:  {auc_lgb:.4f}')
print(f'PR-AUC:   {pr_lgb:.4f}')
print(f'Modell gespeichert: {model_dir / "lgb_recall_risk.pkl"}')

## Zusammenfassung & Nächste Schritte

### Was haben wir gebaut?

Ein LightGBM-Modell, das auf Basis von MAUDE-Reports vorhersagt,
ob ein Gerät innerhalb von 12 Monaten einen FDA Recall haben wird.

### Was macht das Modell gut?
- Rolling-Window Features erfassen Muster über Zeit (nicht nur einzelne Events)
- Severity Score ist domänenbasiert und erklärbar
- SHAP-Values ermöglichen Einzel-Erklärungen
- Modell-Metadaten dokumentieren Limitationen (EU AI Act konform)

### Nächste Schritte (Notebook 04 – FastAPI + Deployment)

1. **FastAPI Endpoint** `/predict`: Nimmt Gerätedaten entgegen, gibt Risk Score + SHAP-Erklärung zurück
2. **RAG Pipeline** `/explain`: Mappt Risk Score auf relevante EU MDR / BfArM Artikel
3. **Streamlit Demo**: Einfaches UI für Präsentation
4. **Docker + Deployment**: Auf Hugging Face Spaces oder Railway

**Regulatory Note:**
Das Modell ist nach EU AI Act Annex III als Hochrisiko-KI-System einzustufen
(automatisierte Entscheidung mit potenziell erheblichen Auswirkungen auf Patientensicherheit).
Die `model_metadata.json` ist ein erster Schritt zur technischen Dokumentation
nach Art. 11 EU AI Act.